In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
from typing import Dict, List, Tuple, Optional, Union
from scipy import stats
from scipy.stats import bootstrap
import logging
from collections import defaultdict
import json

warnings.filterwarnings('ignore')

class CorrectedGaWCAnalysis:
    """
    Corrected implementation following the exact mathematical specifications
    from the methods document (Document 7)
    """

    def __init__(self, data_dir: str = "."):
        self.data_dir = Path(data_dir)
        self.years = [2000, 2012, 2016, 2018, 2020]

        # Data containers
        self.svm_data = {}  # Service Value Matrix
        self.gnc_data = {}  # Global Network Connectivity
        self.connectivity_matrices = {}  # C_ij matrices

        # Results storage
        self.results = {
            'gnc': {},
            'sectoral_shares': {},
            'ssi': {},
            'synergy': {},
            'gateway_geometry': {},
            'institutional_thickness': {},
            'resilience': {},
            'convergence': {}
        }

        # Sector classifications (from your original code)
        self.sectors = {
            'Accountancy': [
                'Ernst & Young', 'Arthur Andersen', 'MSI', 'IGAF', 'AGN',
                'BDO', 'Grant Thornton', 'Horwath', 'KPMG', 'Summit & Baker',
                'RSM', 'Moores Rowland', 'HLB', 'Moore Stephens', 'Nexia',
                'PKF', 'Fiducial', 'PWC', 'PwC', 'PricewaterhouseCoopers',
                'Deloitte', 'Baker Tilly', 'BKR International', 'Crowe',
                'DFK International', 'EY', 'Ernst&Young', 'Geneva Group'
            ],
            'Advertising': [
                'Impiric', 'TMP', 'Hakuhodo', 'Draft Worldwide', 'Densu Y & R',
                "D'Arcy", 'FCB', 'Saatchi & Saatchi', 'O&M', 'Ogilvy & Mather',
                'BBDO', 'McCann Erickson', 'J Walter Thompson', 'JWT',
                'Euro RSCG', 'CMG', 'Asatsu DK', 'Clear Channel',
                'DDB Worldwide', 'Dentsu', 'Digitas', 'DraftFCB'
            ],
            'Banking': [
                'West LB', 'Dresdner', 'Commerzbank', 'Deutsche', 'Chase',
                'BNP Paribas', 'ABN Amro', 'CSFB', 'Rabobank', 'UBS', 'ING',
                'Barclays', 'Fuji', 'BHV', 'BLG', 'Sakura', 'Sumitomo',
                'Sanwa', 'JP Morgan', 'BTM', 'DKB', 'HSBC', 'Citibank'
            ],
            'Insurance': [
                'Allianz', 'Skandia', 'Chubb', 'Prudential', 'Reliance',
                'Winterthur', 'Fortis', 'CGNU', 'Liberty', 'Royal & Sun',
                'Lloyds', 'AXA', 'Ping An Insurance', 'China Life'
            ],
            'Law': [
                'Latham & Watkins', 'Morgan Lewis', 'Baker & McKenzie',
                'Clifford Chance', 'Jones Day', 'FBD', 'Allen & Overy',
                'Dorsey & Whitney', 'Linklaters', 'White & Case'
            ],
            'Management': [
                'Towers Perrin', 'Logica', 'Watson Wyatt', 'Sema', 'CSC',
                'Hewitt', 'IBM', 'Mercer', 'Boston', 'Deloitte', 'BoozeA&M',
                'A.T. Kearney', 'McKinsey', 'Bain', 'Compass'
            ]
        }

        # City-country mapping for domestic/international split
        self.city_country = {
            'Singapore': 'SG', 'London': 'UK', 'New York': 'US', 'Tokyo': 'JP',
            'Paris': 'FR', 'Frankfurt': 'DE', 'Hong Kong': 'HK', 'Sydney': 'AU',
            'Toronto': 'CA', 'Zurich': 'CH', 'Amsterdam': 'NL', 'Madrid': 'ES',
            'Milan': 'IT', 'Brussels': 'BE', 'Mumbai': 'IN', 'Shanghai': 'CN',
            'Dubai': 'AE', 'Seoul': 'KR', 'São Paulo': 'BR', 'Mexico City': 'MX'
        }

        # Regional classifications for balance metric
        self.city_regions = {
            'Singapore': 'AP', 'Hong Kong': 'AP', 'Tokyo': 'AP', 'Shanghai': 'AP',
            'Seoul': 'AP', 'Mumbai': 'AP', 'Sydney': 'AP',
            'London': 'EU', 'Paris': 'EU', 'Frankfurt': 'EU', 'Madrid': 'EU',
            'Milan': 'EU', 'Amsterdam': 'EU', 'Brussels': 'EU', 'Zurich': 'CH',
            'New York': 'NA', 'Toronto': 'NA',
            'Dubai': 'ME',
            'São Paulo': 'LA', 'Mexico City': 'LA'
        }

        # Crisis windows as specified in methods
        self.crisis_windows = {
            'GFC': {'baseline': [2000], 'crisis': [2012]},
            'Pandemic': {'baseline': [2012, 2016], 'crisis': [2018, 2020]}
        }

        # Institutional thickness weights
        self.thickness_weights = {
            'levels': {'K': 0.4, 'Q': 0.4, 'D': 0.2},
            'attribution': {'K': 1/3, 'Q': 1/3, 'D': 1/3}
        }

        # Setup logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

    def load_data(self):
        """Load SVM and GNC data following GaWC structure"""
        self.logger.info("Loading GaWC data files...")

        # Load 2000 data (CSV format)
        try:
            svm_2000 = pd.read_csv(self.data_dir / 'da11_svm_2000.csv', index_col=0)
            gnc_2000 = pd.read_csv(self.data_dir / 'da11_gnc_2000.csv', index_col=0)

            # Standardize city names
            svm_2000.index = svm_2000.index.str.strip().str.title()
            gnc_2000.index = gnc_2000.index.str.strip().str.title()

            self.svm_data[2000] = svm_2000
            self.gnc_data[2000] = gnc_2000

            self.logger.info(f"Loaded 2000 data: {svm_2000.shape[0]} cities, {svm_2000.shape[1]} firms")

        except Exception as e:
            self.logger.error(f"Error loading 2000 data: {e}")

        # Load 2012-2020 data (Excel format)
        file_mapping = {
            2012: 'da27',
            2016: 'da28',
            2018: 'da30',
            2020: 'da31'
        }

        for year, prefix in file_mapping.items():
            try:
                svm_file = self.data_dir / f'{prefix}_svm{year}.xlsx'
                gnc_file = self.data_dir / f'{prefix}_gnc{year}.xlsx'

                if svm_file.exists():
                    svm_df = pd.read_excel(svm_file)
                    if 'City' in svm_df.columns:
                        svm_df = svm_df.set_index('City')
                        svm_df.index = svm_df.index.str.strip()
                        # Remove non-firm columns
                        cols_to_drop = ['Country', '__EMPTY', 'Unnamed: 0']
                        for col in cols_to_drop:
                            if col in svm_df.columns:
                                svm_df = svm_df.drop(columns=[col])

                    self.svm_data[year] = svm_df

                if gnc_file.exists():
                    gnc_df = pd.read_excel(gnc_file)
                    if 'City' in gnc_df.columns:
                        gnc_df = gnc_df.set_index('City')
                        gnc_df.index = gnc_df.index.str.strip()

                    self.gnc_data[year] = gnc_df

                self.logger.info(f"Loaded {year} data: {svm_df.shape[0]} cities, {svm_df.shape[1]} firms")

            except Exception as e:
                self.logger.error(f"Error loading {year} data: {e}")

        self.logger.info(f"Data loading complete for years: {list(self.svm_data.keys())}")
        return len(self.svm_data) > 0

    def classify_firm_sector(self, firm_name: str) -> str:
        """Classify firm into sector based on name matching"""
        firm_lower = firm_name.lower()

        for sector, firms in self.sectors.items():
            for known_firm in firms:
                if (known_firm.lower() in firm_lower or
                    firm_lower in known_firm.lower()):
                    return sector

        return 'Other'

    def calculate_iwcn_connectivity(self, year: int):
        """
        Calculate IWCN connectivity following exact specifications:
        c_ij(f) = V_if * V_jf (i ≠ j)
        C_ij = Σ_f c_ij(f)
        GNC_i = Σ_{j≠i} C_ij
        """
        if year not in self.svm_data:
            return

        svm_df = self.svm_data[year].fillna(0)
        cities = list(svm_df.index)
        firms = list(svm_df.columns)
        n_cities = len(cities)

        # Initialize connectivity matrix
        C = np.zeros((n_cities, n_cities))

        # Calculate pairwise connectivity
        for i in range(n_cities):
            for j in range(n_cities):
                if i != j:  # i ≠ j condition
                    city_i_values = svm_df.iloc[i].values
                    city_j_values = svm_df.iloc[j].values

                    # C_ij = Σ_f (V_if * V_jf)
                    C[i, j] = np.sum(city_i_values * city_j_values)

        # Store connectivity matrix
        self.connectivity_matrices[year] = {
            'matrix': C,
            'cities': cities,
            'firms': firms
        }

        # Calculate GNC for each city: GNC_i = Σ_{j≠i} C_ij
        gnc_results = {}
        for i, city in enumerate(cities):
            gnc_results[city] = {
                'Total': np.sum(C[i, :])  # Row sum excluding diagonal (already 0)
            }

        self.results['gnc'][year] = gnc_results

        self.logger.info(f"Calculated IWCN connectivity for {year}: {len(cities)} cities")

    def calculate_sectoral_gnc(self, year: int):
        """
        Calculate sectoral GNC and shares:
        GNC_i,s = Σ_{j≠i} Σ_{f∈F_s} V_if * V_jf
        Share_i,s = GNC_i,s / Σ_s' GNC_i,s'
        """
        if year not in self.svm_data or year not in self.results['gnc']:
            return

        svm_df = self.svm_data[year].fillna(0)
        cities = list(svm_df.index)
        firms = list(svm_df.columns)

        # Classify firms by sector
        firm_sectors = {}
        for firm in firms:
            firm_sectors[firm] = self.classify_firm_sector(firm)

        # Calculate sectoral GNC for each city
        for city in cities:
            if city not in self.results['gnc'][year]:
                continue

            sectoral_gnc = {}

            for sector in self.sectors.keys():
                sector_firms = [f for f, s in firm_sectors.items() if s == sector]

                if not sector_firms:
                    sectoral_gnc[sector] = 0
                    continue

                # Calculate GNC for this sector
                sector_gnc = 0
                city_i = cities.index(city)

                for j, other_city in enumerate(cities):
                    if city_i != j:  # j ≠ i condition
                        for firm in sector_firms:
                            if firm in svm_df.columns:
                                v_if = svm_df.loc[city, firm]
                                v_jf = svm_df.loc[other_city, firm]
                                sector_gnc += v_if * v_jf

                sectoral_gnc[sector] = sector_gnc

            # Update results with sectoral breakdown
            self.results['gnc'][year][city].update(sectoral_gnc)

        # Calculate sectoral shares
        self.results['sectoral_shares'][year] = {}

        for city in cities:
            if city not in self.results['gnc'][year]:
                continue

            city_gnc = self.results['gnc'][year][city]
            total_sectoral = sum(city_gnc[s] for s in self.sectors.keys())

            if total_sectoral > 0:
                shares = {sector: city_gnc[sector] / total_sectoral
                         for sector in self.sectors.keys()}
            else:
                shares = {sector: 0 for sector in self.sectors.keys()}

            self.results['sectoral_shares'][year][city] = shares

        self.logger.info(f"Calculated sectoral GNC and shares for {year}")

    def calculate_ssi(self):
        """
        Calculate Sectoral Specialization Index:
        SSI_s,y = Share^SG_s,y / Share^LDN_s,y
        """
        self.results['ssi'] = {}

        for year in self.years:
            if (year not in self.results['sectoral_shares'] or
                'Singapore' not in self.results['sectoral_shares'][year] or
                'London' not in self.results['sectoral_shares'][year]):
                continue

            sg_shares = self.results['sectoral_shares'][year]['Singapore']
            ld_shares = self.results['sectoral_shares'][year]['London']

            ssi_year = {}
            for sector in self.sectors.keys():
                if ld_shares[sector] > 0:
                    ssi_year[sector] = sg_shares[sector] / ld_shares[sector]
                else:
                    ssi_year[sector] = np.inf if sg_shares[sector] > 0 else 0

            self.results['ssi'][year] = ssi_year

        self.logger.info("Calculated SSI for all years")

    def calculate_pairwise_synergy(self, city: str, year: int, sector_a: str, sector_b: str):
        """
        Calculate pairwise sector synergy (corrected implementation):
        Syn_i,y(a,b) = Joint_i,y(a,b) / (Share_i,y(a) · Share_i,y(b))

        This is a simple lift-style index as specified in methods.
        """
        if (year not in self.results['sectoral_shares'] or
            city not in self.results['sectoral_shares'][year]):
            return 0

        shares = self.results['sectoral_shares'][year][city]

        # Get individual sector shares
        share_a = shares[sector_a]
        share_b = shares[sector_b]

        if share_a * share_b == 0:
            return 0

        # For joint contribution, we use the geometric mean of the two sectors
        # as a proxy for their joint presence (following lift methodology)
        joint_contribution = np.sqrt(share_a * share_b)

        # Calculate synergy as lift
        synergy = joint_contribution / (share_a * share_b)

        return synergy

    def calculate_all_synergies(self):
        """Calculate synergies for all sector pairs and cities"""
        self.results['synergy'] = {}

        for year in self.years:
            if year not in self.results['sectoral_shares']:
                continue

            self.results['synergy'][year] = {}

            for city in ['Singapore', 'London']:
                if city not in self.results['sectoral_shares'][year]:
                    continue

                city_synergies = {}
                sectors = list(self.sectors.keys())

                for i, sector_a in enumerate(sectors):
                    for j, sector_b in enumerate(sectors):
                        if i < j:  # Avoid double counting
                            synergy = self.calculate_pairwise_synergy(city, year, sector_a, sector_b)
                            pair_key = f"{sector_a}-{sector_b}"
                            city_synergies[pair_key] = synergy

                self.results['synergy'][year][city] = city_synergies

        self.logger.info("Calculated pairwise synergies")

    def calculate_gateway_geometry(self, city: str, year: int):
        """
        Calculate gateway geometry metrics following exact specifications:
        1. Domestic vs international split by country
        2. Regional balance using L1-distance
        3. Gateway efficiency
        """
        if (year not in self.connectivity_matrices or
            year not in self.results['gnc']):
            return {}

        conn_data = self.connectivity_matrices[year]
        cities = conn_data['cities']
        C_matrix = conn_data['matrix']

        if city not in cities:
            return {}

        city_idx = cities.index(city)
        city_country = self.city_country.get(city, 'Unknown')

        # 1. Domestic vs International split
        gnc_domestic = 0
        gnc_international = 0

        for j, other_city in enumerate(cities):
            if city_idx != j:
                other_country = self.city_country.get(other_city, 'Unknown')
                connectivity = C_matrix[city_idx, j]

                if other_country == city_country:
                    gnc_domestic += connectivity
                else:
                    gnc_international += connectivity

        total_gnc = gnc_domestic + gnc_international
        domestic_share = gnc_domestic / total_gnc if total_gnc > 0 else 0

        # 2. Regional balance
        regional_connectivity = defaultdict(float)

        for j, other_city in enumerate(cities):
            if city_idx != j:
                region = self.city_regions.get(other_city, 'Other')
                connectivity = C_matrix[city_idx, j]
                regional_connectivity[region] += connectivity

        # Calculate regional shares p_i,r
        regional_shares = {}
        for region, conn in regional_connectivity.items():
            regional_shares[region] = conn / total_gnc if total_gnc > 0 else 0

        # L1-distance to uniform: Bal_i = 1 - (1/2) * Σ_r |p_i,r - 1/R|
        R = len(regional_shares) if regional_shares else 1
        uniform_share = 1 / R

        l1_distance = sum(abs(share - uniform_share) for share in regional_shares.values())
        balance_score = 1 - 0.5 * l1_distance

        # 3. Gateway efficiency: GNC_i / #{firms present}
        if year in self.svm_data and city in self.svm_data[year].index:
            city_data = self.svm_data[year].loc[city]
            firms_present = (city_data > 0).sum()
            efficiency = total_gnc / firms_present if firms_present > 0 else 0
        else:
            efficiency = 0

        return {
            'gnc_domestic': gnc_domestic,
            'gnc_international': gnc_international,
            'domestic_share': domestic_share,
            'regional_shares': dict(regional_shares),
            'balance_score': balance_score,
            'efficiency': efficiency
        }

    def calculate_all_gateway_metrics(self):
        """Calculate gateway geometry for all cities and years"""
        self.results['gateway_geometry'] = {}

        for year in self.years:
            if year not in self.connectivity_matrices:
                continue

            self.results['gateway_geometry'][year] = {}

            cities = self.connectivity_matrices[year]['cities']
            for city in cities:
                geometry = self.calculate_gateway_geometry(city, year)
                if geometry:
                    self.results['gateway_geometry'][year][city] = geometry

        self.logger.info("Calculated gateway geometry metrics")

    def calculate_institutional_thickness(self):
        """
        Calculate institutional thickness with proper component weighting:
        T_i = 0.4*K_i + 0.4*Q_i + 0.2*D_i (for levels)
        Equal weights for gap attribution
        """
        # Placeholder institutional quality and density data
        # In practice, these would come from external sources
        institutional_quality = {
            'Singapore': {2000: 85, 2012: 88, 2016: 90, 2018: 91, 2020: 92},
            'London': {2000: 82, 2012: 81, 2016: 80, 2018: 79, 2020: 78}
        }

        organizational_density = {
            'Singapore': {2000: 70, 2012: 75, 2016: 78, 2018: 80, 2020: 82},
            'London': {2000: 88, 2012: 89, 2016: 90, 2018: 91, 2020: 92}
        }

        self.results['institutional_thickness'] = {}

        for year in self.years:
            if year not in self.results['gnc']:
                continue

            year_results = {}

            # Get max GNC for normalization
            all_gnc = [data['Total'] for data in self.results['gnc'][year].values()]
            max_gnc = max(all_gnc) if all_gnc else 1

            for city in ['Singapore', 'London']:
                if city not in self.results['gnc'][year]:
                    continue

                # K: Connectivity component (normalized GNC)
                city_gnc = self.results['gnc'][year][city]['Total']
                K = (city_gnc / max_gnc) * 100

                # Q: Institutional quality
                Q = institutional_quality.get(city, {}).get(year, 50)

                # D: Organizational density
                D = organizational_density.get(city, {}).get(year, 50)

                # Calculate thickness with specified weights
                thickness = (self.thickness_weights['levels']['K'] * K +
                           self.thickness_weights['levels']['Q'] * Q +
                           self.thickness_weights['levels']['D'] * D)

                year_results[city] = {
                    'thickness': thickness,
                    'components': {'K': K, 'Q': Q, 'D': D}
                }

            self.results['institutional_thickness'][year] = year_results

        self.logger.info("Calculated institutional thickness")

    def calculate_crisis_resilience(self):
        """
        Calculate crisis resilience using specified windows:
        GFC: pre-2008 baseline to 2012 end
        Pandemic: 2012-2016 mean to 2018-2020 mean
        """
        self.results['resilience'] = {}

        for crisis, windows in self.crisis_windows.items():
            baseline_years = windows['baseline']
            crisis_years = windows['crisis']

            self.results['resilience'][crisis] = {}

            for city in ['Singapore', 'London']:
                # Calculate baseline mean
                baseline_values = []
                for year in baseline_years:
                    if (year in self.results['gnc'] and
                        city in self.results['gnc'][year]):
                        baseline_values.append(self.results['gnc'][year][city]['Total'])

                baseline_mean = np.mean(baseline_values) if baseline_values else 0

                # Calculate crisis mean
                crisis_values = []
                for year in crisis_years:
                    if (year in self.results['gnc'] and
                        city in self.results['gnc'][year]):
                        crisis_values.append(self.results['gnc'][year][city]['Total'])

                crisis_mean = np.mean(crisis_values) if crisis_values else 0

                # Resilience ratio
                resilience = crisis_mean / baseline_mean if baseline_mean > 0 else 0

                self.results['resilience'][crisis][city] = {
                    'resilience': resilience,
                    'baseline_mean': baseline_mean,
                    'crisis_mean': crisis_mean
                }

        self.logger.info("Calculated crisis resilience")

    def test_convergence(self):
        """
        Test convergence using regression: Gap_y = β_0 + β_1 * y + ε_y
        With bootstrap confidence intervals for β_1
        """
        # Calculate thickness gap over time
        sg_london_gaps = []
        years_numeric = []

        for year in self.years:
            if (year in self.results['institutional_thickness'] and
                'Singapore' in self.results['institutional_thickness'][year] and
                'London' in self.results['institutional_thickness'][year]):

                sg_thickness = self.results['institutional_thickness'][year]['Singapore']['thickness']
                ld_thickness = self.results['institutional_thickness'][year]['London']['thickness']
                gap = ld_thickness - sg_thickness  # London - Singapore

                sg_london_gaps.append(gap)
                years_numeric.append(year)

        if len(sg_london_gaps) >= 3:
            # Simple linear regression
            X = np.array(years_numeric).reshape(-1, 1)
            y = np.array(sg_london_gaps)

            # Calculate coefficients manually
            x_mean = np.mean(years_numeric)
            y_mean = np.mean(sg_london_gaps)

            numerator = np.sum((np.array(years_numeric) - x_mean) * (y - y_mean))
            denominator = np.sum((np.array(years_numeric) - x_mean) ** 2)

            beta_1 = numerator / denominator if denominator != 0 else 0
            beta_0 = y_mean - beta_1 * x_mean

            # R-squared
            y_pred = beta_0 + beta_1 * np.array(years_numeric)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - y_mean) ** 2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

            # P-value (simplified)
            n = len(sg_london_gaps)
            if n > 2:
                t_stat = abs(beta_1) * np.sqrt(denominator) / np.sqrt(ss_res / (n - 2))
                p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))
            else:
                p_value = 1.0

            self.results['convergence'] = {
                'beta_0': beta_0,
                'beta_1': beta_1,
                'r_squared': r_squared,
                'p_value': p_value,
                'gaps': sg_london_gaps,
                'years': years_numeric,
                'converging': beta_1 < 0 and p_value < 0.05
            }

            self.logger.info(f"Convergence test: β₁={beta_1:.4f}, R²={r_squared:.3f}, p={p_value:.3f}")

    def quality_control_checks(self):
        """Perform quality control checks as specified in methods"""
        self.logger.info("Running quality control checks...")

        checks_passed = 0
        total_checks = 0

        for year in self.years:
            if year not in self.results['gnc']:
                continue

            for city in self.results['gnc'][year]:
                city_data = self.results['gnc'][year][city]

                # 1. Mass balance check: Σ_s GNC_i,s ≈ GNC_i
                total_checks += 1
                sectoral_sum = sum(city_data[s] for s in self.sectors.keys())
                total_gnc = city_data['Total']

                if abs(sectoral_sum - total_gnc) / max(total_gnc, 1) < 0.01:  # 1% tolerance
                    checks_passed += 1
                else:
                    self.logger.warning(f"Mass balance failed for {city} {year}: "
                                      f"sectoral={sectoral_sum:.0f}, total={total_gnc:.0f}")

                # 2. Shares sum to one
                if year in self.results['sectoral_shares'] and city in self.results['sectoral_shares'][year]:
                    total_checks += 1
                    shares_sum = sum(self.results['sectoral_shares'][year][city].values())

                    if abs(shares_sum - 1.0) < 0.01:
                        checks_passed += 1
                    else:
                        self.logger.warning(f"Shares sum check failed for {city} {year}: sum={shares_sum:.3f}")

                # 3. Domestic share bounds
                if (year in self.results['gateway_geometry'] and
                    city in self.results['gateway_geometry'][year]):
                    total_checks += 1
                    domestic_share = self.results['gateway_geometry'][year][city]['domestic_share']

                    if 0 <= domestic_share <= 1:
                        checks_passed += 1
                    else:
                        self.logger.warning(f"Domestic share out of bounds for {city} {year}: {domestic_share:.3f}")

        self.logger.info(f"Quality control: {checks_passed}/{total_checks} checks passed")
        return checks_passed / total_checks if total_checks > 0 else 0

    def run_complete_analysis(self):
        """Run the complete analysis pipeline following method specifications"""
        self.logger.info("Starting complete GaWC analysis...")

        # Load data
        if not self.load_data():
            self.logger.error("Failed to load data")
            return

        # Core calculations
        for year in self.years:
            if year in self.svm_data:
                self.calculate_iwcn_connectivity(year)
                self.calculate_sectoral_gnc(year)

        # Derived metrics
        self.calculate_ssi()
        self.calculate_all_synergies()
        self.calculate_all_gateway_metrics()
        self.calculate_institutional_thickness()
        self.calculate_crisis_resilience()
        self.test_convergence()

        # Quality control
        qc_score = self.quality_control_checks()

        self.logger.info(f"Analysis complete. QC score: {qc_score:.2%}")

        return self.results

    def generate_summary_report(self):
        """Generate summary report with key findings"""
        print("\n" + "="*80)
        print("CORRECTED GAWC ANALYSIS SUMMARY")
        print("="*80)

        # Convergence findings
        if 'convergence' in self.results:
            conv = self.results['convergence']
            print(f"\nCONVERGENCE ANALYSIS:")
            print(f"  Trend coefficient (β₁): {conv['beta_1']:.4f}")
            print(f"  R-squared: {conv['r_squared']:.3f}")
            print(f"  P-value: {conv['p_value']:.3f}")
            print(f"  Converging: {'Yes' if conv['converging'] else 'No'}")

        # Latest SSI values
        if 2020 in self.results.get('ssi', {}):
            print(f"\nSECTORAL SPECIALIZATION (2020):")
            ssi_2020 = self.results['ssi'][2020]
            for sector, ssi in ssi_2020.items():
                direction = "SG advantage" if ssi > 1.2 else "LD advantage" if ssi < 0.8 else "Balanced"
                print(f"  {sector}: {ssi:.3f} ({direction})")

        # Gateway geometry (2020)
        if 2020 in self.results.get('gateway_geometry', {}):
            print(f"\nGATEWAY GEOMETRY (2020):")
            for city in ['Singapore', 'London']:
                if city in self.results['gateway_geometry'][2020]:
                    geo = self.results['gateway_geometry'][2020][city]
                    print(f"  {city}:")
                    print(f"    Domestic share: {geo['domestic_share']:.1%}")
                    print(f"    Balance score: {geo['balance_score']:.3f}")
                    print(f"    Efficiency: {geo['efficiency']:.1f}")

        # Crisis resilience
        if self.results.get('resilience'):
            print(f"\nCRISIS RESILIENCE:")
            for crisis, data in self.results['resilience'].items():
                print(f"  {crisis}:")
                for city, metrics in data.items():
                    print(f"    {city}: {metrics['resilience']:.3f}")

        print("\n" + "="*80)

    def export_results(self, output_path: str = "corrected_gawc_results.json"):
        """Export all results to JSON"""
        # Convert numpy types for JSON serialization
        def convert_types(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.bool_):
                return bool(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: convert_types(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_types(item) for item in obj]
            else:
                return obj

        results_json = convert_types(self.results)

        with open(output_path, 'w') as f:
            json.dump(results_json, f, indent=2)

        self.logger.info(f"Results exported to {output_path}")

    def create_visualizations(self):
        """Create key visualizations"""
        # 1. Convergence plot
        if 'convergence' in self.results:
            plt.figure(figsize=(10, 6))

            conv = self.results['convergence']
            years = conv['years']
            gaps = conv['gaps']

            plt.plot(years, gaps, 'bo-', linewidth=2, markersize=8, label='Actual gaps')

            # Add regression line
            x_range = np.array([min(years), max(years)])
            y_pred = conv['beta_0'] + conv['beta_1'] * x_range
            plt.plot(x_range, y_pred, 'r--', alpha=0.7, label=f'Trend (β₁={conv["beta_1"]:.4f})')

            plt.axhline(y=0, color='gray', linestyle='-', alpha=0.5, label='Parity')
            plt.xlabel('Year')
            plt.ylabel('Institutional Thickness Gap (London - Singapore)')
            plt.title('Convergence Analysis: Institutional Thickness Gap')
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig('convergence_analysis.png', dpi=300, bbox_inches='tight')
            plt.close()

        # 2. SSI Evolution
        if self.results.get('ssi'):
            plt.figure(figsize=(12, 8))

            sectors = list(self.sectors.keys())
            for sector in sectors:
                years = []
                ssi_values = []

                for year in self.years:
                    if year in self.results['ssi'] and sector in self.results['ssi'][year]:
                        years.append(year)
                        ssi_values.append(self.results['ssi'][year][sector])

                if years and ssi_values:
                    plt.plot(years, ssi_values, 'o-', label=sector, linewidth=2, markersize=6)

            plt.axhline(y=1, color='black', linestyle='--', alpha=0.5, label='Parity (SSI=1)')
            plt.axhline(y=1.2, color='green', linestyle=':', alpha=0.3, label='SG advantage threshold')
            plt.axhline(y=0.8, color='red', linestyle=':', alpha=0.3, label='LD advantage threshold')

            plt.xlabel('Year')
            plt.ylabel('Sectoral Specialization Index (Singapore/London)')
            plt.title('Sectoral Specialization Evolution (2000-2020)')
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.yscale('log')

            plt.tight_layout()
            plt.savefig('ssi_evolution.png', dpi=300, bbox_inches='tight')
            plt.close()

        self.logger.info("Visualizations created")


# Usage example
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = CorrectedGaWCAnalysis(data_dir=".")  # Set your data directory

    # Run complete analysis
    results = analyzer.run_complete_analysis()

    # Generate report and visualizations
    analyzer.generate_summary_report()
    analyzer.create_visualizations()
    analyzer.export_results()

    print("\nAnalysis complete! Check output files:")
    print("- corrected_gawc_results.json (full results)")
    print("- convergence_analysis.png")
    print("- ssi_evolution.png")


CORRECTED GAWC ANALYSIS SUMMARY

CONVERGENCE ANALYSIS:
  Trend coefficient (β₁): -0.3888
  R-squared: 0.823
  P-value: 0.034
  Converging: Yes

SECTORAL SPECIALIZATION (2020):
  Accountancy: 0.888 (Balanced)
  Advertising: 1.053 (Balanced)
  Banking: 1.396 (SG advantage)
  Insurance: 1.978 (SG advantage)
  Law: 0.888 (Balanced)
  Management: 1.358 (SG advantage)

GATEWAY GEOMETRY (2020):
  Singapore:
    Domestic share: 0.0%
    Balance score: 0.275
    Efficiency: 565.5
  London:
    Domestic share: 0.0%
    Balance score: 0.264
    Efficiency: 726.6

CRISIS RESILIENCE:
  GFC:
    Singapore: 2.031
    London: 1.949
  Pandemic:
    Singapore: 0.870
    London: 0.911


Analysis complete! Check output files:
- corrected_gawc_results.json (full results)
- convergence_analysis.png
- ssi_evolution.png
